# Prepare matched Tractor-Mix / SAIGE pilot inputs

Builds a **shared recommended-model-complete** analysis cohort for four matched
calibration runs (Tractor/SAIGE × limited/full covariates) from
`covariates.source_rebuilt.csv.gz`:

- `analysis_samples.txt` — VCF ∩ phenotype ∩ covariate IDs, VCF order, full-model complete
- `pheno_cov.tsv` — `ID`, both covariate sets, and selected binary phenotypes (workflow input)
- `covariates_limited.tsv` — standalone `ID` + limited covariate matrix
- `covariates_full.tsv` — standalone `ID` + recommended full covariate matrix
- `covariate_columns_limited.txt` — `sex`, `age`, `PC1`–`PC10`
- `covariate_columns_full.txt` — limited + standardized `coverage` + GC dummies
- `technical_covariate_levels.tsv` — GC reference/dummy columns and coverage mean/sd

Here, **full** means the recommended production set. It intentionally excludes
extraction method, platform, methylation caller, and SV counts; use those only
in explicit sensitivity analyses.

FLARE VCF URIs are resolved from the Terra data table `aou_lr_chrom` via the
firecloud API (column `model_chr_anc_vcf`). Workspace:
`allofus-drc-wgs-LR-prodData` / `AoU_DRC_LongReads_PhaseTwo_Storage`.

Phenotypes: `gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/saige/AoU_Phase2_Phenotype.csv.gz`

Confirmed FLARE ancestries: `eas=0,amr=1,eur=2,afr=3,sas=4` → `num_ancs=5`.

Upload the source-rebuilt covariate table and set `TRACTOR_COVARIATES_GCS` to its URI before running this notebook.

**Terra WDL:** After this notebook, submit `TractorMixPilot.wdl` with Docker
`tractor-mix-pilot:0.4.2` (shared `docker` input). FitNull is R (`fit_null.R`, ~3h/phenotype);
Score is `tractor-mix-score --threads 8` (~30–45 min/phenotype on chr22). The final
cell stages required WDL scripts to `$WORKSPACE_BUCKET/scripts/` — do not use legacy
`fit_null_and_score.R`.

## Two selections

The notebook runs `select_phenotypes.py` twice on the same cohort.

1. **Pilot** → `tractor_mix_pilot_inputs/`, uploaded to `tractor_mix_pilot/`.
   The top `N_PHENOTYPES` by power, unchanged. Workflows in flight read these
   objects, so nothing here is modified.
2. **Full phecode list** → `felix_wgs_inputs/`, uploaded to `felix/phenotypes/wgs/`.
   Every phenotype passing `--min-cases`, with `--sex-specific restrict` so a
   sex-specific phecode does not carry the other sex as controls. Set
   `PHECODE_SEX_GCS` to the PhecodeX definitions (`data/phecodeX_info.csv` in
   the manuscript repo, uploaded once to `$WORKSPACE_BUCKET/refs/`); without
   it the call falls back to the observed case sex ratio.

A check between them asserts that `analysis_samples.txt` and both covariate
column lists are byte-identical. The cohort is fixed by covariate completeness
before any phenotype is selected, so it must not move when only the phenotype
count changes — and an unchanged cohort is what lets the genome-wide run reuse
the cached GRM and all 22 chromosome packs.

Batch the full list into run-table rows with `scripts/make_felix_run_table.py`;
see `felix/configs/terra/RUN_TABLE.md`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook(
    "workspace_paths.py",
    "resolve_flare_uris.py",
    "select_phenotypes.py",
)
from workspace_paths import data_root

ROOT = data_root()
select_py = SCRIPTS / "select_phenotypes.py"
assert select_py.is_file(), select_py
assert "covariates_limited.tsv" in select_py.read_text(), (
    f"{select_py} is outdated (no covariates_limited.tsv)."
)
print("ROOT:", ROOT)
print("scripts:", sorted(p.name for p in SCRIPTS.glob("*")))


In [ ]:
from pathlib import Path
import json
import os

import pandas as pd

from resolve_flare_uris import (
    DEFAULT_ENTITY_TYPE,
    DEFAULT_NAMESPACE,
    DEFAULT_WORKSPACE,
    fetch_chrom_table_firecloud,
    resolve_uris,
)

OUT_DIR = Path("tractor_mix_pilot_inputs")
OUT_DIR.mkdir(exist_ok=True)

# Terra workspace hosting the aou_lr_chrom data table
TERRA_NAMESPACE = os.environ.get("TERRA_NAMESPACE", DEFAULT_NAMESPACE)
TERRA_WORKSPACE = os.environ.get("TERRA_WORKSPACE", DEFAULT_WORKSPACE)
TERRA_ENTITY_TYPE = os.environ.get("TERRA_ENTITY_TYPE", DEFAULT_ENTITY_TYPE)

# FLARE URIs for the GRM marker set and for the sample list.
#
# These are pinned rather than resolved. Two reasons. The GRM is built from
# them and every step-1 null is cache-keyed to that GRM, so the URIs must not
# drift. And `aou_lr_chrom` lives in a storage workspace we do not control: on
# 2026-10-12 a FlareByPopulation run there left chr22's `model_chr_anc_vcf`
# empty while populating `global_anc`, and the resolver — correctly — refused
# to continue. Nothing was deleted; the row was rewritten by someone else's
# submission.
#
# The sample list is read from the chr22 entry, which is what the resolver used
# to return, so `analysis_samples.txt` reproduces exactly.
PINNED_GRM_VCFS = [
    "gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/"
    "c70fc078-7d46-4da1-8f59-30a7ebd4b738/flare/"
    "81ea1db7-94da-4a6b-b869-35b292d2f9a1/call-flare_task/"
    "aou_lr_phase2_v1.chr1.anc.vcf.gz",
    "gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/"
    "aac26765-3585-4399-b888-ff8521112d20/flare/"
    "b13bb7e0-4684-4b4a-a8b3-a4d94e7f48c4/call-flare_task/"
    "aou_lr_phase2_v1.chr22.anc.vcf.gz",
]

FLARE_VCF = os.environ.get("TRACTOR_FLARE_VCF", "") or PINNED_GRM_VCFS[1]
GRM_VCFS = (
    json.loads(os.environ["TRACTOR_GRM_VCFS"])
    if os.environ.get("TRACTOR_GRM_VCFS")
    else list(PINNED_GRM_VCFS)
)
uris = {"flare_vcf": FLARE_VCF, "grm_vcfs": GRM_VCFS, "source": "pinned"}

# Set RESOLVE_FLARE_URIS=1 to go back to the data table, e.g. when adopting a
# new FLARE run. Check the result against the pins before rebuilding the GRM.
if os.environ.get("RESOLVE_FLARE_URIS"):
    print(f"Fetching {TERRA_ENTITY_TYPE} from {TERRA_NAMESPACE}/{TERRA_WORKSPACE} ...")
    chrom_table = fetch_chrom_table_firecloud(
        TERRA_NAMESPACE, TERRA_WORKSPACE, TERRA_ENTITY_TYPE
    )
    print(f"Loaded {len(chrom_table)} chromosomes: {sorted(chrom_table)}")
    uris = resolve_uris(
        chrom_table,
        scan_chrom="chr22",
        grm_chroms=["chr1", "chr22"],
        uri_column="model_chr_anc_vcf",
    )
    uris["source"] = TERRA_ENTITY_TYPE
    FLARE_VCF = os.environ.get("TRACTOR_FLARE_VCF", "") or uris["flare_vcf"]
    GRM_VCFS = uris["grm_vcfs"]
    if GRM_VCFS != PINNED_GRM_VCFS:
        print(
            "WARNING: resolved GRM VCFs differ from the pins. Rebuilding the GRM "
            "invalidates every cached null and step-2 shard:\n"
            + "\n".join(f"  pinned   {u}" for u in PINNED_GRM_VCFS)
            + "\n"
            + "\n".join(f"  resolved {u}" for u in GRM_VCFS)
        )

(OUT_DIR / "flare_uris.json").write_text(json.dumps(uris, indent=2) + "\n")

PHENOTYPE_GCS = os.environ.get(
    "TRACTOR_PHENOTYPE_GCS",
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/saige/AoU_Phase2_Phenotype.csv.gz",
)
PHENOTYPE_CSV = Path("AoU_Phase2_Phenotype.csv.gz")
# Upload the source-rebuilt covariate table to this URI before running on Terra,
# or set TRACTOR_COVARIATES_GCS to its location. Do not fall back to covariates.v2.
COVARIATES_GCS = os.environ.get(
    "TRACTOR_COVARIATES_GCS",
    f"{os.environ.get('WORKSPACE_BUCKET', '').rstrip('/')}/covariates/covariates.source_rebuilt.csv.gz"
    if os.environ.get("WORKSPACE_BUCKET")
    else "",
)
COVARIATES_CSV = Path("covariates.source_rebuilt.csv.gz")
# ##ANCESTRY=<eas=0,amr=1,eur=2,afr=3,sas=4>
NUM_ANCS = int(os.environ.get("TRACTOR_NUM_ANCS", "5"))
N_PHENOTYPES = 10
MIN_CASES = 100
N_PCS = 10

print("FLARE_VCF (chr22 scan):", FLARE_VCF)
print("GRM_VCFS (chr1+chr22):")
for u in GRM_VCFS:
    print(" ", u)
print("PHENOTYPE_GCS:", PHENOTYPE_GCS)
print("COVARIATES_CSV:", COVARIATES_CSV)
print("NUM_ANCS:", NUM_ANCS)
print("OUT_DIR:", OUT_DIR.resolve())
display(uris)

In [ ]:
def sh(cmd: str) -> None:
    print(cmd)
    rc = get_ipython().system(cmd)
    if rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


ws = os.environ.get("WORKSPACE_BUCKET", "")

# Pull phenotype / source-rebuilt covariates into the notebook cwd if needed.
# The URI is intentionally explicit: silently using the legacy v2 table would
# invalidate the rebuilt-table contract.
if not COVARIATES_CSV.exists() and COVARIATES_GCS:
    sh(f"gsutil cp {COVARIATES_GCS} {COVARIATES_CSV}")

if not PHENOTYPE_CSV.exists():
    sh(f"gsutil cp {PHENOTYPE_GCS} {PHENOTYPE_CSV}")

assert COVARIATES_CSV.exists(), (
    f"Missing {COVARIATES_CSV}. Upload the rebuilt table and set "
    "TRACTOR_COVARIATES_GCS=gs://.../covariates.source_rebuilt.csv.gz, "
    "or place it in the notebook working directory."
)
assert PHENOTYPE_CSV.exists(), f"Missing {PHENOTYPE_CSV}"

# Sample IDs from FLARE VCF (stream; do not copy full VCF locally)
vcf_samples = OUT_DIR / "vcf_samples.txt"
if FLARE_VCF.startswith("gs://"):
    sh(f"gsutil cat {FLARE_VCF} | bcftools query -l > {vcf_samples}")
else:
    sh(f"bcftools query -l {FLARE_VCF} > {vcf_samples}")
print(f"Wrote {vcf_samples} ({sum(1 for _ in open(vcf_samples)):,} samples)")

In [ ]:
select_py = SCRIPTS / "select_phenotypes.py"
assert select_py.exists(), select_py

sh(
    f"python3 {select_py} "
    f"--phenotype-csv {PHENOTYPE_CSV} "
    f"--covariates-csv {COVARIATES_CSV} "
    f"--vcf-samples {vcf_samples} "
    f"--out-dir {OUT_DIR} "
    f"--n-phenotypes {N_PHENOTYPES} "
    f"--min-cases {MIN_CASES} "
    f"--n-pcs {N_PCS}"
)

stats = pd.read_csv(OUT_DIR / "selected_phenotype_stats.tsv", sep="\t")
display(stats)
pheno_cov = pd.read_csv(OUT_DIR / "pheno_cov.tsv", sep="\t", nrows=5)
display(pheno_cov)
limited_matrix = pd.read_csv(OUT_DIR / "covariates_limited.tsv", sep="\t", nrows=5)
full_matrix = pd.read_csv(OUT_DIR / "covariates_full.tsv", sep="\t", nrows=5)
display(limited_matrix)
display(full_matrix)
print("limited covariates:", (OUT_DIR / "covariate_columns_limited.txt").read_text().split())
print("full covariates:", (OUT_DIR / "covariate_columns_full.txt").read_text().split())
print("legacy covariate_columns:", (OUT_DIR / "covariate_columns.txt").read_text().split())
levels = pd.read_csv(OUT_DIR / "technical_covariate_levels.tsv", sep="\t")
display(levels)
n_samples = sum(1 for _ in open(OUT_DIR / "analysis_samples.txt"))
print(f"shared full-complete analysis samples: {n_samples:,}")
print(f"num_ancs (for WDL inputs): {NUM_ANCS}")

In [ ]:
# ---------------------------------------------------------------------------
# Full phecode list for the genome-wide FELIX run.
#
# A second, independent selection on the same cohort. The pilot outputs above
# are read by workflows that may still be running and are never overwritten:
# this writes its own directory and uploads to its own bucket prefix.
# ---------------------------------------------------------------------------
WGS_OUT_DIR = Path("felix_wgs_inputs")
WGS_OUT_DIR.mkdir(exist_ok=True)

WGS_MIN_CASES = MIN_CASES                              # same case floor as the pilot
WGS_NESTING = os.environ.get("WGS_NESTING", "keep")    # keep | leaves | roots
WGS_MAX_R2 = os.environ.get("WGS_MAX_R2", "")          # e.g. "0.5"; empty = off

# PhecodeX definitions, carrying the Male/Female/Both sex column. Authoritative
# for --sex-specific restrict; without it select_phenotypes.py falls back to the
# observed case sex ratio at --sex-case-fraction 0.99, a guess rather than a
# definition, and it decides whether a female-only phecode carries men as
# controls. 3,612 codes, 320 of them sex-specific (255 Female, 65 Male).
#
# This is data/phecodeX_info.csv from the manuscript repo. Upload it once:
#   gsutil cp data/phecodeX_info.csv $WORKSPACE_BUCKET/refs/phecodeX_info.csv
PHECODE_SEX_GCS = os.environ.get(
    "PHECODE_SEX_GCS", f"{ws}/refs/phecodeX_info.csv" if ws else ""
)
PHECODE_SEX_TSV = Path("phecodeX_info.csv")
if PHECODE_SEX_GCS and not PHECODE_SEX_TSV.exists():
    if get_ipython().system(f"gsutil cp {PHECODE_SEX_GCS} {PHECODE_SEX_TSV}"):
        print(f"could not fetch {PHECODE_SEX_GCS}; falling back to the case sex ratio")

wgs_cmd = (
    f"python3 {select_py} "
    f"--phenotype-csv {PHENOTYPE_CSV} "
    f"--covariates-csv {COVARIATES_CSV} "
    f"--vcf-samples {vcf_samples} "
    f"--out-dir {WGS_OUT_DIR} "
    f"--n-phenotypes 0 "
    f"--min-cases {WGS_MIN_CASES} "
    f"--n-pcs {N_PCS} "
    f"--sex-specific restrict "
    f"--nesting {WGS_NESTING}"
)
if PHECODE_SEX_TSV.exists():
    wgs_cmd += f" --phecode-sex-tsv {PHECODE_SEX_TSV}"
else:
    print(
        "WARNING: no phecode definitions table, so sex-specific phecodes will be\n"
        "called from the observed case sex ratio. Set PHECODE_SEX_GCS and re-run\n"
        "before treating the sex-specific results as final."
    )
if WGS_MAX_R2:
    wgs_cmd += f" --max-r2 {WGS_MAX_R2}"

sh(wgs_cmd)


In [ ]:
import hashlib


def _md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


# The cohort comes from covariate completeness, before phenotype selection, so
# changing only the phenotype count must not move it. If it does, the cached
# GRM and all 22 chromosome packs are invalidated and the run is not cheap.
pilot_md5 = _md5(OUT_DIR / "analysis_samples.txt")
wgs_md5 = _md5(WGS_OUT_DIR / "analysis_samples.txt")
assert pilot_md5 == wgs_md5, (
    f"analysis_samples.txt differs between the two selections ({pilot_md5} vs "
    f"{wgs_md5}). Find out why before submitting: a different cohort means a "
    "different GRM, so nothing call-caches."
)
print(f"analysis_samples.txt identical to the pilot  [{pilot_md5}]")

for name in ("covariate_columns_limited.txt", "covariate_columns_full.txt"):
    assert _md5(OUT_DIR / name) == _md5(WGS_OUT_DIR / name), (
        f"{name} differs between the two selections; the full list would be a "
        "different model from the pilot."
    )
print("covariate column lists identical to the pilot")

wgs_phenos = (WGS_OUT_DIR / "selected_phenotypes.txt").read_text().split()
pilot_phenos = (OUT_DIR / "selected_phenotypes.txt").read_text().split()
assert len(set(wgs_phenos)) == len(wgs_phenos), "duplicate phenotype names"
print(f"{len(wgs_phenos):,} phenotypes selected ({len(pilot_phenos)} in the pilot)")
dropped = [p for p in pilot_phenos if p not in set(wgs_phenos)]
print("pilot phenotypes absent from the full list:", dropped or "none")

wgs_stats = pd.read_csv(WGS_OUT_DIR / "selected_phenotype_stats.tsv", sep="\t")
if "sex_specific" in wgs_stats.columns:
    ss = wgs_stats[wgs_stats["sex_specific"].astype(str).str.lower() == "true"]
    basis = dict(ss["sex_specific_basis"].value_counts()) if len(ss) else {}
    print(f"sex-specific and restricted: {len(ss):,}   basis: {basis}")
    show = [c for c in ("phenotype", "sex_restricted_to", "sex_specific_basis",
                        "case_sex_fraction", "n_cases", "n_controls")
            if c in wgs_stats.columns]
    if len(ss):
        display(ss[show].head(20))

n_batches = -(-len(wgs_phenos) // 95)
print(f"\nCromwell calls: {71 * len(wgs_phenos) + 92:,} as one workflow; "
      f"{n_batches} batches of 95 is about {71 * 95 + 92:,} each.")
print("Next: scripts/make_felix_run_table.py — see felix/configs/terra/RUN_TABLE.md")
display(wgs_stats.head())


In [ ]:
# Upload prep outputs to the workspace bucket for Terra / Cromwell
samples_path = OUT_DIR / "analysis_samples.txt"
n_upload = sum(1 for line in samples_path.read_text().splitlines() if line.strip())
assert n_upload > 1000, (
    f"Refusing upload: {samples_path} has only {n_upload} IDs "
    f"({samples_path.stat().st_size} bytes). Re-run select_phenotypes first."
)

if ws:
    dest = f"{ws}/tractor_mix_pilot/"
    for name in [
        "analysis_samples.txt",
        "pheno_cov.tsv",
        "covariates_limited.tsv",
        "covariates_full.tsv",
        "selected_phenotypes.txt",
        "selected_phenotype_stats.tsv",
        "covariate_columns_limited.txt",
        "covariate_columns_full.txt",
        "covariate_columns.txt",
        "technical_covariate_levels.tsv",
        "vcf_samples.txt",
        "flare_uris.json",
    ]:
        path = OUT_DIR / name
        if path.exists():
            sh(f"gsutil cp {path} {dest}{name}")
    sh(f"gsutil ls -l {dest}analysis_samples.txt")
    sh(f"gsutil cat {dest}analysis_samples.txt | wc -l")
    print("Uploaded to", dest)
    print("Use the same shared cohort for all four matched runs:")
    print(f"  analysis_samples:        {dest}analysis_samples.txt")
    print(f"  pheno_cov:               {dest}pheno_cov.tsv")
    print(f"  selected_phenotypes:     {dest}selected_phenotypes.txt")
    print(f"  covariates_limited:      {dest}covariates_limited.tsv")
    print(f"  covariates_full:         {dest}covariates_full.tsv")
    print(f"  columns (limited):       {dest}covariate_columns_limited.txt")
    print(f"  columns (full):          {dest}covariate_columns_full.txt")
    print(f"  technical levels:        {dest}technical_covariate_levels.tsv")
    print("Tractor-Mix / SAIGE: use pheno_cov.tsv plus the matching covariate_columns file.")
else:
    print("WORKSPACE_BUCKET unset; outputs left in", OUT_DIR.resolve())

In [ ]:
# Upload the full-list inputs under their own prefix. tractor_mix_pilot/ is
# deliberately untouched: workflows in flight read those objects.
WGS_PREFIX = os.environ.get("WGS_PREFIX", "felix/phenotypes/wgs").strip("/")

wgs_samples = WGS_OUT_DIR / "analysis_samples.txt"
n_wgs = sum(1 for line in wgs_samples.read_text().splitlines() if line.strip())
assert n_wgs > 1000, f"Refusing upload: {wgs_samples} has only {n_wgs} IDs"

if ws:
    wgs_dest = f"{ws}/{WGS_PREFIX}/"
    for name in [
        "analysis_samples.txt",
        "pheno_cov.tsv",
        "selected_phenotypes.txt",
        "selected_phenotype_stats.tsv",
        "phenotype_selection_audit.tsv",
        "covariates_limited.tsv",
        "covariates_full.tsv",
        "covariate_columns_limited.txt",
        "covariate_columns_full.txt",
        "technical_covariate_levels.tsv",
    ]:
        path = WGS_OUT_DIR / name
        if path.exists():
            sh(f"gsutil cp {path} {wgs_dest}{name}")
    print("Uploaded to", wgs_dest)
    print("Cells for the felix_runs rows:")
    print(f"  pheno_cov:           {wgs_dest}pheno_cov.tsv")
    print(f"  analysis_samples:    {wgs_dest}analysis_samples.txt")
    print(f"  covariate_columns:   {wgs_dest}covariate_columns_limited.txt")
    print(f"                       {wgs_dest}covariate_columns_full.txt")
    print("  selected_phenotypes: written per batch by make_felix_run_table.py")
else:
    print("WORKSPACE_BUCKET unset; outputs left in", WGS_OUT_DIR.resolve())


In [ ]:
# Stage WDL scripts for TractorMixPilot / SaigePilot (fit_null.R + helpers)
WDL_SCRIPTS = [
    "fit_null.R",
    "fit_saige_null.R",
    "reorder_dosages.py",
    "sparsify_grm.R",
    "make_plink_keep.py",
    "run_saige_step2.R",
    "build_saige_plink_and_grm.sh",
    "compare_calibration.py",
    "plot_tractor_results.py",
    "resolve_flare_uris.py",
    "select_phenotypes.py",
    "terra_notebook.py",
    "workspace_paths.py",
]

if ws:
    script_dest = f"{ws}/scripts/"
    for name in WDL_SCRIPTS:
        local = SCRIPTS / name
        assert local.is_file(), f"missing {local} — sync repo scripts/ first"
        sh(f"gsutil cp {local} {script_dest}{name}")
    sh(f"gsutil ls -l {script_dest}fit_null.R")
    print("Staged WDL scripts to", script_dest)
    print("TractorMixPilot: FitNull -> fit_null.R; Score -> tractor-mix-score --threads 8 (docker 0.4.2)")
    print("Input JSON: TractorMixPilot.fit_null_script =", f"{script_dest}fit_null.R")
else:
    print("WORKSPACE_BUCKET unset; skip WDL script upload")